## Event detection model results by part of speech

In [1]:
#!pip install --user numpy==1.26.4

In [2]:
#!pip show torch

In [3]:
#!pip install --user torch==2.3.1

In [4]:
#!pip install --user estnltk==1.7.3
#!pip install --user torch==2.4.0
#!pip install --user sentencepiece==0.2.0

In [ ]:
import os
#from estnltk import Text
from estnltk_neural.taggers import BertMorphTagger
from estnltk.converters import json_to_text
from collections import Counter
from nervaluate import Evaluator

#### Initializing BERT-based morphology tagger

In [2]:
model_path ="C:/Users/liivas/Documents/Magistritoo/est-roberta-vm-morph-tagging"

In [3]:
morph_tagger = BertMorphTagger(model_location=model_path, output_layer='morph_analysis', disambiguate=True)

#### Initializing methods

In [ ]:
# JSON-formaadis failide lugemiseks
def read_json_texts(file_path):
    texts = []
    for filename in os.listdir(file_path):
        text_obj = json_to_text(file=file_path + filename)
        texts.append(text_obj)
        
    return texts


def collect_gold_standard_tags(texts_list):
    pos_tags = []
    gold_event_word_spans = []

    for text in texts_list:
        for ev_word in text.gold_word_events:
            if ev_word.nertag != 'O':
                word = text.words.get(ev_word[0])
                gold_event_word_spans.append(word)
                if "A" in word.morph_analysis.partofspeech and "V" in word.morph_analysis.partofspeech:
                    #pos_tags.append("_".join(["|".join(word.morph_analysis.partofspeech), "|".join(word.morph_analysis.form)]))
                    pos_tags.append("A_kesks")
                else:
                    pos_tags.append(word.morph_analysis.partofspeech[0])
                
    return pos_tags, gold_event_word_spans


def collect_pred_tags(texts_list, bert_layer_name):
    A_pos = ['A', 'U', 'C']
    S_pos = ['S', 'H']
    V_pos = ['V']

    gold_tags_S = []
    gold_tags_V = []
    gold_tags_A_kesks = []
    gold_tags_A = []
    bert_tags_S = []
    bert_tags_V = []
    bert_tags_A_kesks = []
    bert_tags_A = []
    
    def get_pred_nertag(text, word, bert_layer_name):
        if bert_layer_name == "estbert_tokens_of_words":
            for w in text.estbert_tokens_of_words:
                if word == text.words.get(w):
                    return w.nertag
        elif bert_layer_name == "estroberta_tokens_of_words":
            for w in text.estroberta_tokens_of_words:
                if word == text.words.get(w):
                    return w.nertag

    for text in texts_list:
        for ev_word in text.gold_word_events: # kuldstandardi kiht
            word = text.words.get(ev_word[0])
            if "A" in word.morph_analysis.partofspeech and "V" in word.morph_analysis.partofspeech:
                gold_tags_A_kesks.append(ev_word.nertag)
                bert_tags_A_kesks.append(get_pred_nertag(text, word, bert_layer_name))
            else:
                if word.morph_analysis.partofspeech[0] in A_pos:
                    gold_tags_A.append(ev_word.nertag)
                    bert_tags_A.append(get_pred_nertag(text, word, bert_layer_name))
                if word.morph_analysis.partofspeech[0] in S_pos:
                    gold_tags_S.append(ev_word.nertag)
                    bert_tags_S.append(get_pred_nertag(text, word, bert_layer_name))
                if word.morph_analysis.partofspeech[0] in V_pos:
                    gold_tags_V.append(ev_word.nertag)
                    bert_tags_V.append(get_pred_nertag(text, word, bert_layer_name))
                    
                    
    return gold_tags_S, gold_tags_V, gold_tags_A_kesks, gold_tags_A, bert_tags_S, bert_tags_V, bert_tags_A_kesks, bert_tags_A

                                   

### Temporal facts corpus

In [ ]:
# ajafaktide korpuse eri žanrite tekstid, millele on märgendatud 
# EstBERT ja Est-RoBERTa mudelite ennustuste märgenduskihid
temp_fact_news_path = 'temporal_facts_corpus_json_news_pred/'
temp_fact_rkogu_path = 'temporal_facts_corpus_json_rkogu_pred/'
temp_fact_horisont_path = 'temporal_facts_corpus_json_horisont_pred/'

In [7]:
news_texts = read_json_texts(temp_fact_news_path)
rkogu_texts = read_json_texts(temp_fact_rkogu_path)
horisont_texts = read_json_texts(temp_fact_horisont_path)

In [14]:
print(len(news_texts))
print(len(rkogu_texts))
print(len(horisont_texts))

77
3
5


In [12]:
news_texts[0]

Text(text='Üleeilne päev oli ilmselt üks rõõmsamaid lehelugejatele ja masendavamaid ajakirjandusele .\nKümned tuhanded tallinlased leidsid sel päeval postkastist läikpaberil teate , et nad saavad ajalehte Postimees tellida üle nelja korra odavamalt kui näiteks tartlased .\nNii reetis Postimees oma kõige ustavamad toetajad : Tartu ja tartlased .\nEnt tasuta lõunaid kahjuks pole ja varem või hiljem peab lugeja ikka kauba eest maksma .\nPostimehe pöörase allahindluse taga on soov suretada teised lehed välja ja saavutada monopoolne seisund .\nSiis saab kergeusklikult tellijalt võtta mitmekordselt tagasi summa , mis neile praegu kingitakse .\nSelle asemel , et kulutada raha parema lehe tegemiseks ning pakkuda tellijale õige hinna eest võimalikult head kaupa , kulutab Postimees raha konkurentide väljasuretamiseks altvöölöökidega .\nSee ei ole ainuüksi rumal , vaid ka ohtlik , sest ajakirjandusmonopol ohustab meie kõigi sõnavabadust .\nLoomulikult ei saa Eesti Päevalehe toimetus , käed rüpes , pealt vaadata , kuidas Postimehe omanik , kes pole kordagi Eestis käinud , püüab Tallinnas tekitada samasugust ühe lehe umbset tõemonopoli , nagu see on neil korda läinud Tartus .\nMeil ei olnud valikut , me olime sunnitud kaitsma oma lugeja vabadust valida lehte sisu , mitte hinna järgi .\nSeepärast pakume kuni 22. juunini Tallinna , Harjumaa ja Tartu elanikele Eesti Päevalehe nelja kuu tellimust 60 krooni eest ( seni 288 krooni ) ja kuue kuu tellimust 95 krooni eest ( seni 432 krooni ) .\nMe saame aru , et see on ebaõiglane teiste piirkondade inimeste ja seniste , lehe eest täishinda maksnud tellijate suhtes , kuid niisuguse sundkäigu surus meile peale Postimehe alatus .\nEbaõigluse leevendamiseks otsustasime neil , kellel on 31. juuli seisuga tellitud Eesti Päevaleht täishinnaga , pikendada tellimust tasuta ühe kuu võrra kogu Eestis .\nTellija ei pea ise midagi tegema , toimetus saadab talle pärast tellimuse lõppu automaatselt veel kuu aja jooksul iga päev lehe koju .\nPostimehe vallandatud hinnasõda , mis lugejale võib esmapilgul tunduda suure õnnena , on tegelikult meie kõigi ühine õnnetus .\nEesti Päevalehe toimetus maksab üliodavate tellimuste ja juba olemasolevate tellimuste tasuta pikendamise eest soolast hinda ja ei saa seetõttu võibolla ellu viia kõiki plaane , mis olid kavandatud lehe arendamiseks .\nEnt mingis mõttes on see hind siiski madal , sest sõnavabadus ja lugeja valikuvabadus on hindamatu väärtus .\n')

In [ ]:
# BERT-i taggeriga tekstidele morfo peale märgendamine
for text in news_texts:
    morph_tagger.retag( text )
    
for text in rkogu_texts:
    morph_tagger.retag( text )
    
for text in horisont_texts:
    morph_tagger.retag( text )

c:\Users\liivas\AppData\Local\anaconda3\envs\py310\lib\site-packages\estnltk_neural\taggers\embeddings\bert\bert_tokens_to_words_rewriter.py:192: UserWarning: (!) No matching words span for bert token Span(' ', [{'bert_tokens': '▁', 'form': 'pl n', 'partofspeech': 'C', 'probability': 0.99877}]).
  warnings.warn(f"(!) No matching {words_layer.name} span for bert token {bert_tokens_layer[i]}.")
c:\Users\liivas\AppData\Local\anaconda3\envs\py310\lib\site-packages\estnltk_neural\taggers\embeddings\bert\bert_tokens_to_words_rewriter.py:192: UserWarning: (!) No matching words span for bert token Span(' ', [{'bert_tokens': '▁', 'form': 'sg p', 'partofspeech': 'S', 'probability': 0.99979}]).
  warnings.warn(f"(!) No matching {words_layer.name} span for bert token {bert_tokens_layer[i]}.")


In [32]:
# kuldstandard-sündmused ja sõnaliigid
news_gold_pos_tags, news_gold_event_word_spans = collect_gold_standard_tags(news_texts)
rkogu_gold_pos_tags, rkogu_gold_event_word_spans = collect_gold_standard_tags(rkogu_texts)
horisont_gold_pos_tags, horisont_gold_event_word_spans = collect_gold_standard_tags(horisont_texts)

In [ ]:
# sõnaliigid ja nende sagedused eri žanrites enne kesksõnade eraldamist
#print(f"Uudised: {Counter(news_gold_pos_tags).most_common()}")
#print(f"Stenogrammid: {Counter(rkogu_gold_pos_tags).most_common()}")
#print(f"Ajalugu: {Counter(horisont_gold_pos_tags).most_common()}")

Uudised: [('V', 582), ('S', 397), ('A', 71), ('N', 9), ('D', 8), ('H', 8), ('C', 6), ('Z', 6), ('O', 4), ('U', 4), ('Y', 3), ('G', 2), ('K', 2)]
Stenogrammid: [('V', 56), ('S', 48), ('A', 7), ('O', 2), ('Y', 1), ('N', 1)]
Ajalugu: [('V', 56), ('S', 30), ('A', 11), ('C', 2), ('H', 1)]


In [ ]:
# sõnaliigid ja nende sagedused eri žanrites, vaatame kesksõnadele viitavate juhtude morfoinfot
#print(f"Uudised: {Counter(news_gold_pos_tags).most_common()}")
#print(f"Stenogrammid: {Counter(rkogu_gold_pos_tags).most_common()}")
#print(f"Ajalugu: {Counter(horisont_gold_pos_tags).most_common()}")

Uudised: [('V', 570), ('S', 397), ('A', 69), ('N', 9), ('D', 8), ('H', 8), ('V|A|A|A_nud||sg n|pl n', 7), ('C', 6), ('Z', 6), ('O', 4), ('U', 4), ('Y', 3), ('V|A|A|A_nud|pl n||sg n', 2), ('G', 2), ('K', 2), ('V|A|A|A_nud||pl n|sg n', 1), ('A|V|A|A_|nud|pl n|sg n', 1), ('V|A|A|A_nud|pl n|sg n|', 1), ('A|V|A|A_|tud|sg n|pl n', 1), ('V|A|A|A_nud|sg n|pl n|', 1)]
Stenogrammid: [('V', 56), ('S', 48), ('A', 7), ('O', 2), ('Y', 1), ('N', 1)]
Ajalugu: [('V', 55), ('S', 30), ('A', 11), ('C', 2), ('H', 1), ('V|A|A|A_nud||sg n|pl n', 1)]


In [ ]:
# sõnaliigid ja nende sagedused eri žanrites, kesksõnad eraldatud
print(f"Uudised: {Counter(news_gold_pos_tags).most_common()}")
print(f"Stenogrammid: {Counter(rkogu_gold_pos_tags).most_common()}")
print(f"Ajalugu: {Counter(horisont_gold_pos_tags).most_common()}")

Uudised: [('V', 570), ('S', 397), ('A', 69), ('A_kesks', 14), ('N', 9), ('D', 8), ('H', 8), ('C', 6), ('Z', 6), ('O', 4), ('U', 4), ('Y', 3), ('G', 2), ('K', 2)]
Stenogrammid: [('V', 56), ('S', 48), ('A', 7), ('O', 2), ('Y', 1), ('N', 1)]
Ajalugu: [('V', 55), ('S', 30), ('A', 11), ('C', 2), ('H', 1), ('A_kesks', 1)]


In [ ]:
# korpusest leitud sõnaliikide Vabamorfi tagid:
"""
U - omadussõna (ülivõrre)
G - genitiivatribuut (käändumatu omadussõna)
S - nimisõna
K - kaassõna
Y - lühend
D - määrsõna
A - omadussõna (algvõrre)
V - verb
N - põhiarvsõna
H - pärisnimi
C - omadussõna (keskvõrre)
O - järgarvsõna
Z - kirjavahemärk/sümbol
"""

# sõnaliigikide tagide agregeerimise põhimõtted:
# verbid -> V
# omadussõnad -> A, C, U
# nimisõnad -> S, H
# kesksõnad -> A_kesks

In [34]:
print(len(news_gold_event_word_spans))
print(len(rkogu_gold_event_word_spans))
print(len(horisont_gold_event_word_spans))

1102
115
100


### EstBERT ajafaktide uudistel

In [37]:
news_gold_tags_S, news_gold_tags_V, news_gold_tags_A_kesks, news_gold_tags_A, news_estbert_tags_S, news_estbert_tags_V, news_estbert_tags_A_kesks, news_estbert_tags_A = collect_pred_tags(news_texts, "estbert_tokens_of_words")

In [38]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([news_gold_tags_S], [news_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

In [39]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 197,
 'incorrect': 9,
 'partial': 0,
 'missed': 179,
 'spurious': 1687,
 'possible': 385,
 'actual': 1893,
 'precision': 0.10406761753829899,
 'recall': 0.5116883116883116,
 'f1': 0.17295873573309922}

In [40]:
# Omadussõnade tulemus
nervaluate_A = Evaluator([news_gold_tags_A], [news_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

In [ ]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 54,
 'incorrect': 4,
 'partial': 0,
 'missed': 21,
 'spurious': 434,
 'possible': 79,
 'actual': 492,
 'precision': 0.10975609756097561,
 'recall': 0.6835443037974683,
 'f1': 0.18914185639229422}

In [45]:
# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([news_gold_tags_A_kesks], [news_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

In [46]:
results_A_kesks['strict']

{'correct': 14,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 69,
 'possible': 14,
 'actual': 83,
 'precision': 0.1686746987951807,
 'recall': 1.0,
 'f1': 0.288659793814433}

In [42]:
# Verbide tulemus
nervaluate_V = Evaluator([news_gold_tags_V], [news_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [43]:
results_V['strict']

{'correct': 563,
 'incorrect': 2,
 'partial': 0,
 'missed': 3,
 'spurious': 4344,
 'possible': 568,
 'actual': 4909,
 'precision': 0.1146873090242412,
 'recall': 0.9911971830985915,
 'f1': 0.2055870001825817}

### EstBERT ajafaktide stenogrammidel

In [44]:
rkogu_gold_tags_S, rkogu_gold_tags_V, rkogu_gold_tags_A_kesks, rkogu_gold_tags_A, rkogu_estbert_tags_S, rkogu_estbert_tags_V, rkogu_estbert_tags_A_kesks, rkogu_estbert_tags_A = collect_pred_tags(rkogu_texts, "estbert_tokens_of_words")

In [47]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([rkogu_gold_tags_S], [rkogu_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

In [48]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 14,
 'incorrect': 0,
 'partial': 0,
 'missed': 33,
 'spurious': 380,
 'possible': 47,
 'actual': 394,
 'precision': 0.03553299492385787,
 'recall': 0.2978723404255319,
 'f1': 0.06349206349206349}

In [49]:
# Omadussõnade tulemus
nervaluate_A = Evaluator([rkogu_gold_tags_A], [rkogu_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

In [50]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 3,
 'incorrect': 0,
 'partial': 0,
 'missed': 4,
 'spurious': 83,
 'possible': 7,
 'actual': 86,
 'precision': 0.03488372093023256,
 'recall': 0.42857142857142855,
 'f1': 0.06451612903225806}

In [51]:
# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([rkogu_gold_tags_A_kesks], [rkogu_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

In [52]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 0,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 4,
 'possible': 0,
 'actual': 4,
 'precision': 0.0,
 'recall': 0,
 'f1': 0}

In [53]:
# Verbide tulemus
nervaluate_V = Evaluator([rkogu_gold_tags_V], [rkogu_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [54]:
# Verbide tulemus
results_V['strict']

{'correct': 54,
 'incorrect': 2,
 'partial': 0,
 'missed': 0,
 'spurious': 900,
 'possible': 56,
 'actual': 956,
 'precision': 0.056485355648535567,
 'recall': 0.9642857142857143,
 'f1': 0.10671936758893281}

### EstBERT ajafaktide ajalool

In [55]:
horisont_gold_tags_S, horisont_gold_tags_V, horisont_gold_tags_A_kesks, horisont_gold_tags_A, horisont_estbert_tags_S, horisont_estbert_tags_V, horisont_estbert_tags_A_kesks, horisont_estbert_tags_A = collect_pred_tags(horisont_texts, "estbert_tokens_of_words")

In [58]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([horisont_gold_tags_S], [horisont_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

# Omadussõnade tulemus
nervaluate_A = Evaluator([horisont_gold_tags_A], [horisont_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([horisont_gold_tags_A_kesks], [horisont_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

# Verbide tulemus
nervaluate_V = Evaluator([horisont_gold_tags_V], [horisont_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [59]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 11,
 'incorrect': 0,
 'partial': 0,
 'missed': 17,
 'spurious': 147,
 'possible': 28,
 'actual': 158,
 'precision': 0.06962025316455696,
 'recall': 0.39285714285714285,
 'f1': 0.11827956989247311}

In [60]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 9,
 'incorrect': 0,
 'partial': 0,
 'missed': 4,
 'spurious': 87,
 'possible': 13,
 'actual': 96,
 'precision': 0.09375,
 'recall': 0.6923076923076923,
 'f1': 0.16513761467889906}

In [61]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 1,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 8,
 'possible': 1,
 'actual': 9,
 'precision': 0.1111111111111111,
 'recall': 1.0,
 'f1': 0.19999999999999998}

In [62]:
# Verbide tulemus
results_V['strict']

{'correct': 55,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 461,
 'possible': 55,
 'actual': 516,
 'precision': 0.1065891472868217,
 'recall': 1.0,
 'f1': 0.19264448336252188}

### Est-RoBERTa ajafaktide uudistel

In [63]:
news_gold_tags_S, news_gold_tags_V, news_gold_tags_A_kesks, news_gold_tags_A, news_estbert_tags_S, news_estbert_tags_V, news_estbert_tags_A_kesks, news_estbert_tags_A = collect_pred_tags(news_texts, "estroberta_tokens_of_words")

In [64]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([news_gold_tags_S], [news_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

# Omadussõnade tulemus
nervaluate_A = Evaluator([news_gold_tags_A], [news_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([news_gold_tags_A_kesks], [news_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

# Verbide tulemus
nervaluate_V = Evaluator([news_gold_tags_V], [news_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [65]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 241,
 'incorrect': 11,
 'partial': 0,
 'missed': 133,
 'spurious': 1942,
 'possible': 385,
 'actual': 2194,
 'precision': 0.10984503190519598,
 'recall': 0.625974025974026,
 'f1': 0.18689414501744864}

In [66]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 54,
 'incorrect': 4,
 'partial': 0,
 'missed': 21,
 'spurious': 430,
 'possible': 79,
 'actual': 488,
 'precision': 0.11065573770491803,
 'recall': 0.6835443037974683,
 'f1': 0.1904761904761905}

In [67]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 14,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 68,
 'possible': 14,
 'actual': 82,
 'precision': 0.17073170731707318,
 'recall': 1.0,
 'f1': 0.2916666666666667}

In [68]:
# Verbide tulemus
results_V['strict']

{'correct': 559,
 'incorrect': 9,
 'partial': 0,
 'missed': 0,
 'spurious': 4338,
 'possible': 568,
 'actual': 4906,
 'precision': 0.11394211169995923,
 'recall': 0.9841549295774648,
 'f1': 0.2042382170259408}

### Est-RoBERTa ajafaktide stenogrammidel

In [69]:
rkogu_gold_tags_S, rkogu_gold_tags_V, rkogu_gold_tags_A_kesks, rkogu_gold_tags_A, rkogu_estbert_tags_S, rkogu_estbert_tags_V, rkogu_estbert_tags_A_kesks, rkogu_estbert_tags_A = collect_pred_tags(rkogu_texts, "estroberta_tokens_of_words")

In [70]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([rkogu_gold_tags_S], [rkogu_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

# Omadussõnade tulemus
nervaluate_A = Evaluator([rkogu_gold_tags_A], [rkogu_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([rkogu_gold_tags_A_kesks], [rkogu_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

# Verbide tulemus
nervaluate_V = Evaluator([rkogu_gold_tags_V], [rkogu_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [71]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 21,
 'incorrect': 0,
 'partial': 0,
 'missed': 26,
 'spurious': 467,
 'possible': 47,
 'actual': 488,
 'precision': 0.0430327868852459,
 'recall': 0.44680851063829785,
 'f1': 0.07850467289719625}

In [72]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 5,
 'incorrect': 0,
 'partial': 0,
 'missed': 2,
 'spurious': 68,
 'possible': 7,
 'actual': 73,
 'precision': 0.0684931506849315,
 'recall': 0.7142857142857143,
 'f1': 0.125}

In [73]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 0,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 3,
 'possible': 0,
 'actual': 3,
 'precision': 0.0,
 'recall': 0,
 'f1': 0}

In [74]:
# Verbide tulemus
results_V['strict']

{'correct': 48,
 'incorrect': 0,
 'partial': 0,
 'missed': 8,
 'spurious': 925,
 'possible': 56,
 'actual': 973,
 'precision': 0.04933196300102775,
 'recall': 0.8571428571428571,
 'f1': 0.09329446064139942}

### Est-RoBERTa ajafaktide ajalool

In [75]:
horisont_gold_tags_S, horisont_gold_tags_V, horisont_gold_tags_A_kesks, horisont_gold_tags_A, horisont_estbert_tags_S, horisont_estbert_tags_V, horisont_estbert_tags_A_kesks, horisont_estbert_tags_A = collect_pred_tags(horisont_texts, "estroberta_tokens_of_words")

In [76]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([horisont_gold_tags_S], [horisont_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

# Omadussõnade tulemus
nervaluate_A = Evaluator([horisont_gold_tags_A], [horisont_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([horisont_gold_tags_A_kesks], [horisont_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

# Verbide tulemus
nervaluate_V = Evaluator([horisont_gold_tags_V], [horisont_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [77]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 13,
 'incorrect': 1,
 'partial': 0,
 'missed': 14,
 'spurious': 163,
 'possible': 28,
 'actual': 177,
 'precision': 0.07344632768361582,
 'recall': 0.4642857142857143,
 'f1': 0.12682926829268293}

In [78]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 9,
 'incorrect': 0,
 'partial': 0,
 'missed': 4,
 'spurious': 86,
 'possible': 13,
 'actual': 95,
 'precision': 0.09473684210526316,
 'recall': 0.6923076923076923,
 'f1': 0.16666666666666669}

In [79]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 1,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 8,
 'possible': 1,
 'actual': 9,
 'precision': 0.1111111111111111,
 'recall': 1.0,
 'f1': 0.19999999999999998}

In [80]:
# Verbide tulemus
results_V['strict']

{'correct': 55,
 'incorrect': 0,
 'partial': 0,
 'missed': 0,
 'spurious': 466,
 'possible': 55,
 'actual': 521,
 'precision': 0.10556621880998081,
 'recall': 1.0,
 'f1': 0.1909722222222222}

### TimeML corpus

### EstBERT TimeML lõplikul testhulgal (uudised)

In [81]:
timeml_final_test_path = 'timeml_final_test_pred/'

timeml_texts = read_json_texts(timeml_final_test_path)
    

In [82]:
for text in timeml_texts:
    morph_tagger.retag( text )

In [89]:
# kuldstandard-sündmused
timeml_gold_pos_tags, timeml_gold_event_word_spans = collect_gold_standard_tags(timeml_texts)

In [90]:
# sõnaliigid ja nende sagedused timeml lõplikus testhulgas
print(f"TimeML lõplik testhulk: {Counter(timeml_gold_pos_tags).most_common()}")

TimeML lõplik testhulk: [('V', 206), ('S', 107), ('A', 22), ('H', 4), ('A_kesks', 3), ('Y', 3), ('J', 2), ('X', 1), ('C', 1), ('D', 1)]


In [92]:
print(len(timeml_texts))

8


In [93]:
print(len(timeml_gold_event_word_spans))

350


In [91]:
timeml_gold_tags_S, timeml_gold_tags_V, timeml_gold_tags_A_kesks, timeml_gold_tags_A, timeml_estbert_tags_S, timeml_estbert_tags_V, timeml_estbert_tags_A_kesks, timeml_estbert_tags_A = collect_pred_tags(timeml_texts, "estbert_tokens_of_words")

In [94]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([timeml_gold_tags_S], [timeml_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

# Omadussõnade tulemus
nervaluate_A = Evaluator([timeml_gold_tags_A], [timeml_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([timeml_gold_tags_A_kesks], [timeml_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

# Verbide tulemus
nervaluate_V = Evaluator([timeml_gold_tags_V], [timeml_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [95]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 81,
 'incorrect': 1,
 'partial': 0,
 'missed': 26,
 'spurious': 24,
 'possible': 108,
 'actual': 106,
 'precision': 0.7641509433962265,
 'recall': 0.75,
 'f1': 0.7570093457943926}

In [96]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 19,
 'incorrect': 1,
 'partial': 0,
 'missed': 2,
 'spurious': 10,
 'possible': 22,
 'actual': 30,
 'precision': 0.6333333333333333,
 'recall': 0.8636363636363636,
 'f1': 0.7307692307692307}

In [97]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 1,
 'incorrect': 1,
 'partial': 0,
 'missed': 0,
 'spurious': 1,
 'possible': 2,
 'actual': 3,
 'precision': 0.3333333333333333,
 'recall': 0.5,
 'f1': 0.4}

In [98]:
# Verbide tulemus
results_V['strict']

{'correct': 189,
 'incorrect': 6,
 'partial': 0,
 'missed': 5,
 'spurious': 14,
 'possible': 200,
 'actual': 209,
 'precision': 0.9043062200956937,
 'recall': 0.945,
 'f1': 0.9242053789731052}

### Est-RoBERTa TimeML lõplikul testhulgal (uudised)

In [99]:
timeml_gold_tags_S, timeml_gold_tags_V, timeml_gold_tags_A_kesks, timeml_gold_tags_A, timeml_estbert_tags_S, timeml_estbert_tags_V, timeml_estbert_tags_A_kesks, timeml_estbert_tags_A = collect_pred_tags(timeml_texts, "estroberta_tokens_of_words")

In [100]:
# Nimisõnade tulemus
nervaluate_S = Evaluator([timeml_gold_tags_S], [timeml_estbert_tags_S], tags=['EVENT'], loader='list')
results_S, results_by_tag_S, result_indices_S, result_indices_by_tag_S = nervaluate_S.evaluate()

# Omadussõnade tulemus
nervaluate_A = Evaluator([timeml_gold_tags_A], [timeml_estbert_tags_A], tags=['EVENT'], loader='list')
results_A, results_by_tag_A, result_indices_A, result_indices_by_tag_A = nervaluate_A.evaluate()

# Kesksõnade tulemus
nervaluate_A_kesks = Evaluator([timeml_gold_tags_A_kesks], [timeml_estbert_tags_A_kesks], tags=['EVENT'], loader='list')
results_A_kesks, results_by_tag_A_kesks, result_indices_A_kesks, result_indices_by_tag_A_kesks = nervaluate_A_kesks.evaluate()

# Verbide tulemus
nervaluate_V = Evaluator([timeml_gold_tags_V], [timeml_estbert_tags_V], tags=['EVENT'], loader='list')
results_V, results_by_tag_V, result_indices_V, result_indices_by_tag_V = nervaluate_V.evaluate()

In [101]:
# Nimisõnade tulemus
results_S['strict']

{'correct': 92,
 'incorrect': 1,
 'partial': 0,
 'missed': 15,
 'spurious': 20,
 'possible': 108,
 'actual': 113,
 'precision': 0.8141592920353983,
 'recall': 0.8518518518518519,
 'f1': 0.832579185520362}

In [102]:
# Omadussõnade tulemus
results_A['strict']

{'correct': 19,
 'incorrect': 1,
 'partial': 0,
 'missed': 2,
 'spurious': 10,
 'possible': 22,
 'actual': 30,
 'precision': 0.6333333333333333,
 'recall': 0.8636363636363636,
 'f1': 0.7307692307692307}

In [104]:
# Kesksõnade tulemus
results_A_kesks['strict']

{'correct': 1,
 'incorrect': 1,
 'partial': 0,
 'missed': 0,
 'spurious': 1,
 'possible': 2,
 'actual': 3,
 'precision': 0.3333333333333333,
 'recall': 0.5,
 'f1': 0.4}

In [105]:
# Verbide tulemus
results_V['strict']

{'correct': 188,
 'incorrect': 7,
 'partial': 0,
 'missed': 5,
 'spurious': 14,
 'possible': 200,
 'actual': 209,
 'precision': 0.8995215311004785,
 'recall': 0.94,
 'f1': 0.9193154034229829}